# ✍️ Lab 2.5: The Digit Recognizer

In Lab 2, data augmentation was a hero: flipping cats gave us free data and beat overfitting. 💪


## 📚 The plan
1. 🔢 Build & train a digit CNN on **MNIST** (handwritten 0–9)
2. 😈 Re-train it with a BAD augmentation 
3. 🖊️ Draw your own digits online and see if YOUR model reads them

⚡ **Recommended:** GPU runtime.

📦 **Dataset:** MNIST — downloads automatically via torchvision.

In [ ]:
!pip install torch torchvision matplotlib numpy tqdm pillow

### 🛠️ Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", "✅ GPU" if device.type == 'cuda' else "❌ CPU (works, just slower)")

### 📂 Load MNIST — Handwritten Digits 0–9

MNIST is 70,000 grayscale 28×28 images of handwritten digits. It's the "hello world" of computer vision. Notice the digits are **white on a black background** and **centered** — remember this for the drawing part later! 🖤🤍

In [ ]:
transform = transforms.ToTensor()

trainset = torchvision.datasets.MNIST(root='./data', train=True,  download=True, transform=transform)
testset  = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader  = torch.utils.data.DataLoader(testset,  batch_size=64, shuffle=False)

print(f"🔢 Training images: {len(trainset)}, Test images: {len(testset)}")

images, labels = next(iter(trainloader))
fig, axes = plt.subplots(1, 10, figsize=(15, 2))
for ax, img, lbl in zip(axes, images[:10], labels[:10]):
    ax.imshow(img.squeeze(), cmap='gray'); ax.axis('off'); ax.set_title(int(lbl))
plt.suptitle('White digits on black, centered — the MNIST look 🖤')
plt.show()

### 🏗️ Section 1: Build & Train the Digit CNN

Same recipe as Lab 2 (conv → pool → conv → pool → dense), adjusted for grayscale (1 channel in) and 10 classes out.

**📐 The architecture you're about to build:**


![My Image](./Structures/lab2.png)

*This is the same diagram from the theory session — now let's code it.*

**📐 Read this diagram carefully — it tells you every number you need:**



**How to read it:** each block is a layer. Under each block, the numbers tell you its settings.
- `1 → 16 | 3×3, pad 1` means: **Conv** layer, **1** channel in, **16** filters out, **3×3** kernel, padding **1**.
- `2 × 2` under MaxPool means a **2×2** pooling window.
- `32 × 7 × 7 = 1568` is the **flatten** size (channels × height × width).
- `1568 → 64` and `64 → 10` are the two **FC (Linear)** layers.

In [ ]:
# 🛠️ YOUR TASK: fill in each blank marked with ______ using the diagram above.
# The comments and 💡 hints tell you exactly what number goes where.
# Tip: this is almost identical to Lab 2's CatDogCNN — the only differences are
#      (1) input channels = 1 (grayscale, not 3 color), and (2) output = 10 (digits 0-9).

class DigitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # First conv: takes 1 channel (grayscale) in, produces 16 filters. Kernel 3, padding 1.
        # 💡 Diagram: "1 → 16 | 3×3, pad 1"
        self.conv1 = nn.Conv2d(______, ______, kernel_size=3, padding=1)

        # Second conv: takes the 16 maps from conv1, produces 32. Kernel 3, padding 1.
        # 💡 Diagram: "16 → 32 | 3×3, pad 1"
        self.conv2 = nn.Conv2d(______, ______, kernel_size=3, padding=1)

        # Pooling window is 2×2 (halves the image size each time).
        # 💡 Diagram: "2 × 2"
        self.pool  = nn.MaxPool2d(______, ______)

        # First Linear layer: from the flattened size to 64.
        # 💡 Diagram: "32 × 7 × 7 = 1568"  →  so the input size is 32 * 7 * 7
        self.fc1   = nn.Linear(32 * 7 * 7, ______)

        # Final Linear layer: from 64 down to 10 (one score per digit 0-9).
        # 💡 Diagram: "64 → 10"
        self.fc2   = nn.Linear(______, ______)

    def forward(self, x):
        # The forward pass is already written for you — it matches Lab 2 exactly.
        # Read it to see how the layers connect: conv → relu → pool, twice, then flatten, then dense.
        x = self.pool(F.relu(self.conv1(x)))   # 28 → 14
        x = self.pool(F.relu(self.conv2(x)))   # 14 → 7
        x = x.view(-1, 32 * 7 * 7)             # flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Quick self-check: if you filled everything correctly, this runs with NO error
# and prints "✅ Shape looks correct!"
try:
    _test = DigitCNN()
    _out = _test(torch.zeros(1, 1, 28, 28))   # one fake 28x28 grayscale image
    assert _out.shape == (1, 10), f"Output shape is {_out.shape}, expected (1, 10)"
    print("✅ Shape looks correct! Your network outputs 10 numbers (one per digit).")
except Exception as e:
    print("❌ Not quite yet:", e)
    print("   Re-check your blanks against the diagram. The solution is in the next cell if you're stuck.")

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            preds = model(images).argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

def train(model, loader, epochs=3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        model.train()
        for images, labels in tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}  |  test accuracy {evaluate(model, testloader)*100:.2f}%")
    return model

good_model = DigitCNN().to(device)
train(good_model, trainloader, epochs=3)
good_acc = evaluate(good_model, testloader)
print(f"\n✅ Our digit reader scores {good_acc*100:.2f}% — excellent!")

### Section 2: "Let's Add More Augmentation!"

A well-meaning student thinks: *"Augmentation helped in Lab 2 — so MORE augmentation must be even better! Let's flip and heavily rotate the digits too."*

Sounds reasonable. Let's actually do it and measure the result. 🧪

In [ ]:
bad_aug = transforms.Compose([
    transforms.RandomVerticalFlip(p=0.5),       
    transforms.RandomRotation(180),             
    transforms.ToTensor(),
])

# Show what this does to a '6'
six_idx = (trainset.targets == 6).nonzero()[0].item()
six_pil = transforms.ToPILImage()(trainset[six_idx][0])

fig, axes = plt.subplots(1, 8, figsize=(15, 2.3))
axes[0].imshow(six_pil, cmap='gray'); axes[0].axis('off'); axes[0].set_title('A real 6')
for ax in axes[1:]:
    ax.imshow(bad_aug(six_pil).squeeze(), cmap='gray'); ax.axis('off'); ax.set_title('...6? or 9?? 😵')
plt.suptitle('😈 Watch: a flipped/rotated 6 becomes indistinguishable from a 9')
plt.show()

In [ ]:
# Train a NEW model on the badly-augmented data
trainset_bad = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=bad_aug)
trainloader_bad = torch.utils.data.DataLoader(trainset_bad, batch_size=64, shuffle=True)

bad_model = DigitCNN().to(device)
train(bad_model, trainloader_bad, epochs=3)
bad_acc = evaluate(bad_model, testloader)

print(f"\n✅ Good augmentation model: {good_acc*100:.2f}%")
print(f"😈 Bad augmentation model:  {bad_acc*100:.2f}%")
print(f"📉 We DESTROYED {(good_acc-bad_acc)*100:.1f} points of accuracy by 'adding more augmentation'!")

### 🔍 Section 3: Where Exactly Did It Break? The Confusion Matrix

The overall accuracy dropped — but WHERE? A **confusion matrix** (remember Day 1's mushrooms!) shows which digits get mistaken for which. Watch the 6↔9 cells light up. 🔥

In [ ]:
def confusion_matrix(model):
    model.eval()
    cm = np.zeros((10, 10), dtype=int)
    with torch.no_grad():
        for images, labels in testloader:
            preds = model(images.to(device)).argmax(1).cpu()
            for t, p in zip(labels, preds):
                cm[t, p] += 1
    return cm

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, model, title in [(axes[0], good_model, '✅ Good augmentation'),
                          (axes[1], bad_model, '😈 Bad augmentation')]:
    cm = confusion_matrix(model)
    im = ax.imshow(cm, cmap='Reds')
    ax.set_xticks(range(10)); ax.set_yticks(range(10))
    ax.set_xlabel('Model predicted'); ax.set_ylabel('True digit'); ax.set_title(title)
    for i in range(10):
        for j in range(10):
            if cm[i, j] > 0:
                ax.text(j, i, cm[i, j], ha='center', va='center',
                        color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=7)
plt.suptitle('🔍 Look at the (6,9) and (9,6) cells on the right — that\'s the damage!')
plt.show()

In [ ]:
# Zoom in on the specific crime: how often is a true 6 called a 9, and vice versa?
cm_bad = confusion_matrix(bad_model)
print("With BAD augmentation:")
print(f"  True 6 → predicted 9:  {cm_bad[6, 9]} times")
print(f"  True 9 → predicted 6:  {cm_bad[9, 6]} times")
cm_good = confusion_matrix(good_model)
print("\nWith GOOD augmentation (for comparison):")
print(f"  True 6 → predicted 9:  {cm_good[6, 9]} times")
print(f"  True 9 → predicted 6:  {cm_good[9, 6]} times")

**There it is.** Rotating a 6 by 180° literally makes a 9. So when we told the model "a rotated/flipped 6 is still a 6," we were **lying to it** — we forced it to believe 6 and 9 are the same thing. The confusion matrix shows exactly that confusion. 🎯

**The rule:** augmentation must preserve the *meaning* of the label. Flipping a cat → still a cat ✅. Flipping a 6 → a 9 ❌.

### 🎯 Mini Exercise 3.1
Besides 6↔9, which OTHER digit pairs might a vertical flip confuse? Look at the bad confusion matrix for other bright off-diagonal cells. (Think about which digits look like something else upside down.) ✍️

### 🖊️ Section 4: Test YOUR Own Handwriting!

Time to challenge your **good** model with digits YOU draw.

**Steps:**
1. Go to 👉 **https://kleki.com**
2. ⚙️ **Important setup for a fair test:**
   - Make the canvas background **black** (or we auto-invert for you below)
   - Use a **thick white brush**
   - Draw ONE digit, **big and centered**
3. Export as PNG (menu → Save / Export)
4. Upload it in the cell below

*Our code auto-detects if you drew dark-on-light and inverts it to match MNIST's white-on-black style.* 🪄

In [ ]:
from PIL import Image, ImageOps

filenames = ['my_digit.png']


def prepare_digit(path):
    img = Image.open(path).convert('L')          # grayscale
    # MNIST is white digit on black. If the image is mostly light, it's dark-on-light -> invert.
    if np.array(img).mean() > 127:
        img = ImageOps.invert(img)
    img = img.resize((28, 28))
    tensor = transforms.ToTensor()(img).unsqueeze(0)
    return img, tensor

good_model.eval()
for fn in filenames:
    img28, tensor = prepare_digit(fn)
    with torch.no_grad():
        out = good_model(tensor.to(device))
        probs = F.softmax(out, dim=1)[0].cpu()
        pred = int(probs.argmax())

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
    a1.imshow(img28, cmap='gray'); a1.axis('off')
    a1.set_title(f"Model reads this as: {pred}  ({probs[pred]*100:.0f}%)", color='blue')
    a2.bar(range(10), probs.numpy(), color='steelblue')
    a2.set_xticks(range(10)); a2.set_xlabel('digit'); a2.set_ylabel('confidence')
    a2.set_title('How sure is it about each digit?')
    plt.show()

### 🎯 Mini Exercise 4.1 — Try to Fool It 😈
Draw a messy 6, then draw a 6 rotated to look like a 9. Feed both to your **good** model. Does it read them the way YOU intended? Then imagine feeding them to the **bad** model — what would happen?

In [ ]:
# TO-DO: upload a tricky digit and test both good_model and bad_model on it


## The GOLDEN Question 🏆

In Lab 2, adding augmentation **helped**. In Lab 2.5, adding augmentation **hurt**. Same technique, opposite result.

**State the single rule that explains BOTH outcomes. Then: a friend building a model to detect whether an X-ray was taken left-side or right-side wants to use horizontal flip augmentation. Good idea or bad idea? Why?** 🤔